# Exercício 21 — agente de **marketing** para campanhas do **produto mais vendido**

Cenário: rede fictícia «Mercado Central» com histórico de vendas agregado (`dados_vendas_demo.py`). O **agente ReAct** (LangGraph + Gemini) chama *tools* que devolvem JSON — ranking, produto #1 em unidades, ficha por SKU e contexto de marca — e redige um **plano de campanha** fundamentado nesses dados.

**Requisitos:** `GOOGLE_API_KEY` ou `GEMINI_API_KEY` no `.env` na raiz. Opcional: `GEMINI_MODEL_EX21`.

**Arranque:** `./run.sh` nesta pasta.


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

for p in (Path.cwd(), *Path.cwd().parents):
    env = p / ".env"
    if env.is_file():
        load_dotenv(env)
        break

here = Path.cwd().resolve()
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

from agente_marketing_estrela import (
    config_marketing,
    criar_grafo_agente_marketing,
    ultima_resposta_assistente,
)
from dados_vendas_demo import produto_mais_vendido_por_unidades, ranking_por_unidades

print("Ranking (top 3) — unidades:")
print(json.dumps(ranking_por_unidades(3), ensure_ascii=False, indent=2))
print("\nLíder em unidades (ground truth para o agente):")
print(json.dumps(produto_mais_vendido_por_unidades(), ensure_ascii=False, indent=2))


Ranking (top 3) — unidades:
[
  {
    "posicao": 1,
    "sku": "AGUA6L",
    "nome": "Água mineral 6×1,5 L",
    "categoria": "Bebidas",
    "unidades_vendidas": 950,
    "receita_total_eur": "3125.50"
  },
  {
    "posicao": 2,
    "sku": "BOLACH400",
    "nome": "Bolachas integrais 400 g",
    "categoria": "Snacks",
    "unidades_vendidas": 620,
    "receita_total_eur": "1109.80"
  },
  {
    "posicao": 3,
    "sku": "CAFE500",
    "nome": "Café Torrado 500 g — linha «Amanhecer»",
    "categoria": "Bebidas quentes / pequeno-almoço",
    "unidades_vendidas": 600,
    "receita_total_eur": "2938.20"
  }
]

Líder em unidades (ground truth para o agente):
{
  "sku": "AGUA6L",
  "nome": "Água mineral 6×1,5 L",
  "categoria": "Bebidas",
  "preco_referencia_eur": "3.29",
  "pitch_curto": "Pack familiar; forte em volume, margem mais baixa.",
  "unidades_vendidas": 950,
  "receita_total_eur": "3125.50"
}


In [2]:
if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY ou GEMINI_API_KEY no .env da raiz.")

grafico = criar_grafo_agente_marketing()
cfg = config_marketing("thread-ex21-demo")

out = grafico.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Planeia uma campanha de marketing de 4 semanas focada no **produto mais vendido em unidades**. "
                    "Inclui nome da campanha, público, pilares, canais, calendário e KPIs. Usa as ferramentas antes de concluir."
                )
            )
        ]
    },
    cfg,
)

print(ultima_resposta_assistente(out))


## Campanha: "Frescura Essencial"

**Público-alvo:** Famílias e indivíduos preocupados com a hidratação diária e que procuram opções convenientes e económicas. A proposta de valor é oferecer uma solução prática para o consumo de água, com um pack familiar a um preço acessível, promovendo um estilo de vida saudável.

**Pilares de mensagem:**

*   Hidratação para toda a família: Água mineral num formato ideal para o consumo familiar.
*   Qualidade e conveniência: Água pura e fresca, disponível em packs fáceis de transportar e armazenar.
*   Preço acessível: A melhor opção para manter a hidratação diária sem comprometer o orçamento.

**Canais e Tácticas:**

| Canal                  | Táctica                                                                                                                                                                                                                                                                                                               